# Climate Stripes: Preprocess with CDO/NCO, Plot with Python

This is the central case study. It reproduces the source `stripes.ipynb` while making the division of labor explicit.

The source code selects 1981–2010 before calculating annual anomalies, which yields only 30 stripes. Here, 1981–2010 is used as a reference period for the complete 1850–2014 series.

In [ ]:
from pathlib import Path
import os
if Path.cwd().name == 'notebooks': os.chdir('..')

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, TwoSlopeNorm
import numpy as np
import xarray as xr
from workshop import annual_global_anomaly, open_tas

DATA = 'data/demo/tas_demo.nc'

## Python reference workflow

The helper function below follows the same concepts as the source notebook: cosine-latitude weighting, annual means, and a baseline anomaly.

In [ ]:
tas = open_tas(DATA)
python_anomaly = annual_global_anomaly(tas, baseline=(1981, 2010))
python_anomaly.plot(figsize=(10, 3), color='black')
plt.axhline(0, color='0.6', linewidth=0.8)
plt.ylabel('Temperature anomaly (degC)')
plt.title('Python/xarray global annual anomaly')
plt.show()

## CDO/NCO preprocessing workflow

The CDO output is a compact one-dimensional NetCDF series. NCO makes its meaning explicit in the variable name and attributes.

In [ ]:
%%bash
set -euo pipefail
bash scripts/run_cdo_workflow.sh data/demo/tas_demo.nc outputs/stripes/cdo
ncks -m -v tas_anom outputs/stripes/cdo/tas_global_annual_anomaly.nc | head -n 35

## Compare values before plotting

A visual match is not sufficient. Compare all annual values numerically.

In [ ]:
cdo_anomaly = xr.open_dataset(
    'outputs/stripes/cdo/tas_global_annual_anomaly.nc', use_cftime=True
)["tas_anom"].squeeze()
difference = np.abs(cdo_anomaly.values - python_anomaly.values)
print(f'Years compared: {difference.size}')
print(f'Maximum absolute difference: {difference.max():.3e} degC')
print(f'Mean absolute difference: {difference.mean():.3e} degC')
assert difference.max() < 2.0e-4

## Plot the CDO-prepared series with Python

At this stage Python does not need the original monthly latitude–longitude field.

In [ ]:
!python scripts/make_stripes.py outputs/stripes/cdo/tas_global_annual_anomaly.nc outputs/stripes/climate_stripes.png

from IPython.display import Image, display
display(Image('outputs/stripes/climate_stripes.png'))

## Change the reference period

For the hands-on exercise, replace `1981/2010` with `1961/1990` in the CDO workflow and `(1981, 2010)` with `(1961, 1990)` in Python. Re-run the numerical comparison before interpreting the new colors.